<a href="https://colab.research.google.com/github/xavierign/FaCells/blob/ori-s-branch/Training_FaCells.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
model_configurations = [
    {"family": "256-3-.4-ML", "units": 256, "layers": 3, "dropout": 0.4},
    # Add more configurations as needed
]

In [2]:
def build_model_from_config(input_shape, config, num_classes):
    """
    Builds a baseline LSTM model dynamically for multi-label classification.

    Args:
        input_shape: Shape of the input data.
        config: Dictionary containing model configuration.
        num_classes: Number of classes (attributes) for multi-label classification.

    Returns:
        A compiled Keras model.
    """
    inputs = Input(shape=input_shape)
    x = Masking(mask_value=0.0)(inputs)

    # Add LSTM layers
    for _ in range(config.get("layers", 1)):
        x = Bidirectional(LSTM(config["units"], return_sequences=True))(x)
        if config.get("dropout", 0) > 0:
            x = Dropout(config["dropout"])(x)

    # Global average pooling over time steps
    x = GlobalAveragePooling1D()(x)

    # Output layer for multi-label classification
    outputs = Dense(num_classes, activation='sigmoid')(x)

    # Compile model for multi-label classification
    model = Model(inputs, outputs)
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

def build_model_from_config_1D(input_shape, config):
    """
    Builds a baseline LSTM model dynamically based on the configuration.

    Args:
        input_shape: Shape of the input data.
        config: Dictionary containing model configuration.

    Returns:
        A compiled Keras model.
    """
    inputs = Input(shape=input_shape)
    x = Masking(mask_value=0.0)(inputs)

    # Add LSTM layers
    for _ in range(config.get("layers", 1)):
        x = Bidirectional(LSTM(config["units"], return_sequences=True))(x)
        if config.get("dropout", 0) > 0:
            x = Dropout(config["dropout"])(x)

    # Global average pooling over time steps
    x = GlobalAveragePooling1D()(x)

    # Output layer
    outputs = Dense(1, activation='sigmoid')(x)

    # Compile model
    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [4]:
import pandas as pd
import datetime
import pickle
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Bidirectional, GlobalAveragePooling1D, Masking
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

def run_baseline_experiments(configurations, X_train, y_train, model_dir="models/"):
    """
    Runs experiments with multiple baseline model configurations and logs results.

    Args:
        configurations: List of model configurations.
        X_train: Training data.
        y_train: Training labels.
        model_dir: Directory to save trained models.

    Returns:
        DataFrame containing results of all experiments.
    """
    results = []

    for i, config in enumerate(configurations):
        print(f"Training model {i + 1}/{len(configurations)}: {config}")

        # Build model
        model = build_model_from_config(X_train.shape[1:], config,1) #40)

        # Train model
        history = model.fit(
            X_train, y_train,
            epochs=10,
            batch_size=32,
            validation_split=0.2,
            verbose=1
        )

        # Evaluate model
        final_acc = history.history["val_accuracy"][-1]
        model_file = f"{model_dir}/baseline_model_{config['family']}_{i}_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.keras"
        model.save(model_file)

        # Log results
        results.append({
            "Family": config["family"],
            "Units": config["units"],
            "Layers": config["layers"],
            "Dropout": config.get("dropout", 0),
            "Validation Accuracy": final_acc,
            "Model File": model_file
        })

    # Convert results to DataFrame
    results_df = pd.DataFrame(results)
    results_df.to_csv(f"{model_dir}/ml_vINI.csv", index=False)
    return results_df


def load_lstm_input(pickle_file):
    """
    Loads LSTM input data from a pickle file.

    Args:
        pickle_file (str): Path to the pickle file.

    Returns:
        tuple: (sequences, labels) - lists of sequences and corresponding labels.
    """
    with open(pickle_file, 'rb') as f:
        data = pickle.load(f)

    sequences, labels = zip(*data)
    return list(sequences), list(labels)

def prepare_data_1d(sequences, labels, max_length=8000):
    """
    Prepares sequences and labels for LSTM training.

    Args:
        sequences (list): List of sequences (variable-length lists of [x, y, l]).
        labels (list): List of labels (arrays or integers).
        max_length (int): Maximum allowed sequence length.

    Returns:
        tuple: (padded_sequences, mask_value, encoded_labels)
    """
    # Pad or truncate sequences to the specified max length
    mask_value = 0.0
    truncated_sequences = [seq[:max_length] for seq in sequences]
    padded_sequences = pad_sequences(truncated_sequences, padding='post', dtype='float32', value=mask_value)

    # Convert labels to a NumPy array (binary classification)
    label_array = np.array([1 if label == 1 else 0 for label in labels])

    return padded_sequences, mask_value, label_array

def prepare_data(sequences, labels, padding_value=0):
    """
    Prepares LSTM input data with padding and converts labels to binary format.

    Args:
        sequences (list of np.ndarray): List of variable-length sequences.
        labels (list of np.ndarray): List of arrays with labels (e.g., [-1, 1]).
        padding_value (int, optional): Value to use for padding. Defaults to 0.

    Returns:
        tuple: (padded_sequences, mask_value, label_array)
            - padded_sequences: NumPy array of padded sequences.
            - mask_value: Value used for masking.
            - label_array: Binary label matrix.
    """
    # Pad sequences to the same length
    padded_sequences = pad_sequences(sequences, padding='post', value=padding_value)

    # Convert labels to a binary matrix
    label_matrix = np.array([(label + 1) // 2 for label in labels])  # Convert -1 to 0 and 1 to 1

    return padded_sequences, padding_value, label_matrix

#pikle_file = "drive/MyDrive/ml_data/lstm_ml_input_data.pkl" #when it's multilabel#
pickle_file = "drive/MyDrive/ml_data/lstm_input_data.pkl"

# Load the data
sequences, labels = load_lstm_input(pickle_file)


# Prepare the data
padded_sequences, mask_value, label_matrix = prepare_data_1d(sequences, labels)
#Example usage
results_df = run_baseline_experiments(model_configurations, padded_sequences, label_matrix, model_dir="drive/MyDrive/ml_data/")

Training model 1/1: {'family': '256-3-.4-ML', 'units': 256, 'layers': 3, 'dropout': 0.4}
Epoch 1/10
  1/750 ━━━━━━━━━━━━━━━━━━━━ 20:52:51 100s/step - accuracy: 0.3438 - loss: 0.7190

KeyboardInterrupt: 

In [ ]:
# Load results
results_df = pd.read_csv("drive/MyDrive/ml_data/baseline_experiment_results_v2.csv")
print(results_df.sort_values(by="Validation Accuracy", ascending=False))

             Family  Units  Layers  Dropout  Validation Accuracy  \
1   stacked_lstm_v2    128       3      0.3             0.812333   
2      deep_lstm_v2    256       2      0.4             0.812167   
3  balanced_lstm_v2    100       3      0.2             0.805500   
0    simple_lstm_v2     64       3      0.2             0.779500   

                                          Model File  
1  drive/MyDrive/ml_data//baseline_model_stacked_...  
2  drive/MyDrive/ml_data//baseline_model_deep_lst...  
3  drive/MyDrive/ml_data//baseline_model_balanced...  
0  drive/MyDrive/ml_data//baseline_model_simple_l...  


In [ ]:
results_df = pd.read_csv("drive/MyDrive/ml_data/baseline_experiment_results.csv")
print(results_df.sort_values(by="Validation Accuracy", ascending=False))

          Family  Units  Layers  Dropout  Validation Accuracy  \
2      deep_lstm    256       3      0.4             0.820333   
1   stacked_lstm    128       2      0.3             0.793167   
3  balanced_lstm    100       2      0.2             0.792333   
0    simple_lstm     64       1      0.2             0.760333   

                                          Model File  
2  drive/MyDrive/ml_data//baseline_model_deep_lst...  
1  drive/MyDrive/ml_data//baseline_model_stacked_...  
3  drive/MyDrive/ml_data//baseline_model_balanced...  
0  drive/MyDrive/ml_data//baseline_model_simple_l...  


In [ ]:
import pandas as pd
results_df = pd.read_csv("drive/MyDrive/ml_data/baseline_experiment_results_v3.csv")
print(results_df.sort_values(by="Validation Accuracy", ascending=False))

  Family  Units  Layers  Dropout  Validation Accuracy  \
2  256-4    256       4      0.3             0.807167   
0  128-4    128       4      0.3             0.807000   
1   64-4     64       4      0.3             0.795000   

                                          Model File  
2  drive/MyDrive/ml_data//baseline_model_256-4_2_...  
0  drive/MyDrive/ml_data//baseline_model_128-4_0_...  
1  drive/MyDrive/ml_data//baseline_model_64-4_1_2...  
